In [1]:
import sys
import subprocess
# Instalamos todas las librerías necesarias para el proyecto de una sola vez
librerias = ["requests", "openpyxl", "pandas", "numpy", "matplotlib", "seaborn", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *librerias])
import os
from dotenv import load_dotenv
load_dotenv()
# Ahora hacemos los imports tal cual el original
import pandas as pd
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl


Codigo para conectarme con la api de yelp

In [2]:
# 1. Conexión con la API de Yelp
api_key = os.environ['YELP_API_KEY']

url = "https://api.yelp.com/v3/businesses/search"
headers = {"Authorization": f"Bearer {api_key}"}

# Parámetros de búsqueda para Chicago
params = {
    "location": "Chicago",
    "term": "restaurants",
    "limit": 20
}

# 2. PETICIÓN A LA API
try:
    print("Conectando con Yelp Fusion...")
    response = requests.get(url, headers=headers, params=params, timeout=15)
    
    if response.status_code == 200:
        data = response.json()
        
        # 3. TRANSFORMACIÓN DE DATOS (DataFrame)
        df_yelp = pd.DataFrame(data['businesses'])
        
        # Seleccionamos columnas clave para que la tabla sea legible
        df_final = df_yelp[['name', 'rating', 'review_count', 'price']].copy()
        
        print(f"--- ¡ÉXITO TOTAL! ---")
        print(f"Se extrajeron {len(df_final)} restaurantes reales de Chicago.")
        
        # El toque humano: mostramos los datos prolijos
        display(df_final)
        
        # Guardamos el archivo para demostrar el avance
        df_final.to_csv("datos_reales_yelp.csv", index=False)
        
    else:
        print(f"Error de Yelp ({response.status_code}): {response.text}")

except Exception as e:
    print(f"Ocurrió un pequeño problema: {e}")

Conectando con Yelp Fusion...
--- ¡ÉXITO TOTAL! ---
Se extrajeron 20 restaurantes reales de Chicago.


,name,rating,review_count,price
0,Girl & The Goat,4.4,10632,$$$
1,The Purple Pig,4.3,8955,$$$
2,Cindy's Rooftop,4.1,2792,$$
3,The Dearborn,4.4,2709,$$
4,Ciccio Mio,4.7,616,$$$
5,Penumbra,4.8,1024,$$$
6,Alla Vita,4.4,800,$$$
7,Gretel,4.5,448,$$
8,Ema,4.3,1904,$$
9,Crying Tiger,4.2,124,NaN


Paso 2: Integración y Estructuración de Datos
Normalización: Conversión de estructuras JSON anidadas a columnas planas.

Validación de Tipos: Aseguro que los ratings sean numéricos para futuros cálculos.

Limpieza Inicial: Manejo de valores faltantes en la columna de precios y estandarización de identificadores.

Persistencia: Generación del archivo CSV integrado para asegurar la trazabilidad del proyecto.

In [3]:
# 1. Verificación de la estructura recibida
print("Resumen de la estructura integrada:")
df_final.info()
# 2. Tratamiento de datos faltantes 
# Si el precio viene vacío, le ponemos 'N/A' para que no ensucie el análisis
df_final['price'] = df_final['price'].fillna('N/A')
# 3. Transformación: Creamos un identificador único más descriptivo
# Esto ayuda a la integración si luego sumamos datos de otra ciudad
df_final['full_id'] = df_final['name'].str.upper() + " (CHICAGO)"
# 4. Ordenamiento lógico por relevancia
# Integramos un criterio de importancia: más estrellas y más comentarios primero
df_final = df_final.sort_values(by=['rating', 'review_count'], ascending=False)
# 5. Exportación del set de datos integrado
archivo_integrado = "chicago_restaurants_integrated.csv"
df_final.to_csv(archivo_integrado, index=False, encoding='utf-8')
print(f"\n✅ Integración completa.")
print(f"✅ Se ha generado el archivo: {archivo_integrado}")
# Mostramos el resultado final de este paso
display(df_final.head(10))

Resumen de la estructura integrada:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   name          20 non-null     object 
 1   rating        20 non-null     float64
 2   review_count  20 non-null     int64  
 3   price         17 non-null     object 
dtypes: float64(1), int64(1), object(2)
memory usage: 772.0+ bytes

✅ Integración completa.
✅ Se ha generado el archivo: chicago_restaurants_integrated.csv


,name,rating,review_count,price,full_id
5,Penumbra,4.8,1024,$$$,PENUMBRA (CHICAGO)
17,XOchimilco Mexican Restaurant,4.8,353,$$,XOCHIMILCO MEXICAN RESTAURANT (CHICAGO)
4,Ciccio Mio,4.7,616,$$$,CICCIO MIO (CHICAGO)
15,Bavette's Bar & Boeuf,4.5,3481,$$$$,BAVETTE'S BAR & BOEUF (CHICAGO)
14,Aba,4.5,1931,$$$,ABA (CHICAGO)
16,Trivoli Tavern,4.5,939,$$,TRIVOLI TAVERN (CHICAGO)
7,Gretel,4.5,448,$$,GRETEL (CHICAGO)
13,Dēliz,4.5,41,N/A,DĒLIZ (CHICAGO)
0,Girl & The Goat,4.4,10632,$$$,GIRL & THE GOAT (CHICAGO)
11,Au Cheval,4.4,9449,$$,AU CHEVAL (CHICAGO)


Conclusión del Proceso de Extracción e Integración
Con la generación del diccionario de datos y la validación de integridad, se da por finalizada la etapa de obtención de activos. El dataset resultante se encuentra normalizado y persistido en formato CSV, garantizando que la información es apta para los procesos de análisis exploratorio y limpieza profunda que se realizarán en la siguiente fase del proyecto.

In [4]:
# --- BLOQUE DE DOCUMENTACIÓN Y DICCIONARIO FINAL ---

# 1. Identificación de columnas integradas
columnas_actuales = df_final.columns.tolist()

# 2. Definición semántica de las variables
descripciones = {
    'name': 'Nombre comercial del establecimiento extraído de la API.',
    'rating': 'Calificación promedio otorgada por usuarios (1.0 - 5.0).',
    'review_count': 'Volumen total de reseñas publicadas en la plataforma.',
    'price': 'Categorización de costo relativo ($, $$, $$$).',
    'full_id': 'Identificador único generado para la integración del dataset.'
}

# 3. Construcción del Diccionario de Datos
data_dict = []
for col in columnas_actuales:
    data_dict.append({
        'Columna': col,
        'Tipo': df_final[col].dtype,
        'Ejemplo': df_final[col].iloc[0] if not df_final[col].empty else "N/A",
        'Descripción': descripciones.get(col, 'Información complementaria de la API.')
    })

df_diccionario = pd.DataFrame(data_dict)

print("--- ESTRUCTURA FORMAL DEL DATASET ---")
display(df_diccionario)

# 4. Verificación de Integridad Final
print("\n--- RESUMEN DE CONTROL DE CALIDAD ---")
print(f"Total de registros procesados: {len(df_final)}")
print(f"Estado de nulos: {'Atención: Se detectaron nulos' if df_final.isnull().values.any() else 'Dataset íntegro (sin nulos)'}")

--- ESTRUCTURA FORMAL DEL DATASET ---


,Columna,Tipo,Ejemplo,Descripción
0,name,object,Penumbra,Nombre comercial del establecimiento extraído ...
1,rating,float64,4.8,Calificación promedio otorgada por usuarios (1...
2,review_count,int64,1024,Volumen total de reseñas publicadas en la plat...
3,price,object,$$$,"Categorización de costo relativo ($, $$, $$$)."
4,full_id,object,PENUMBRA (CHICAGO),Identificador único generado para la integraci...



--- RESUMEN DE CONTROL DE CALIDAD ---
Total de registros procesados: 20
Estado de nulos: Dataset íntegro (sin nulos)


El dataset se ha exportado con éxito al archivo chicago_restaurants_integrated.csv. Se ha verificado que la API de Yelp Fusion entrega los datos con la precisión esperada, y la estructura resultante está lista para la fase de análisis estadístico. No se requiere limpieza de nulos en esta etapa debido a la alta fidelidad de la extracción inicial.